# Area-matched random-mask control

Does attention intervention on **WeDetect-Uni regions** change GeoCLIP more than the
same intervention on an **arbitrary region of the same area**? Section 4 of the paper
leaves this open; this notebook answers it.

For every Img2GPS3k image with an active mask, the Table 1 configuration
(layers {2, 7, 12}, $a=+2/3$, $b=-2/3$) runs three ways in one batch:

| Arm | Mask | Controls for |
|---|---|---|
| `union` | the exact Table 1 mask: cached WeDetect-Uni proposals, score ≥ 0.4, ≤ 100 boxes | — |
| `shift` | the same mask translated to a random offset (wrap-around), ×5 | area, shape, contiguity → isolates **content** |
| `scatter` | the same number of patches drawn uniformly, ×5 | area only |

All logic lives in `experiments/eval_random_mask_control_colab.py`; this notebook
only sets it up and reads the result. Use a **GPU runtime** (G4 / RTX PRO 6000 to match Table 1).

In [ ]:
import torch
print(torch.cuda.get_device_name(0), torch.__version__)

## 1. Setup

Clones the GeoCLIP fork at the commit pinned by the `geoprobe` submodule, and takes
`intervention.py`, the Table 1 proposal cache and the evaluation script from `geoprobe`.
If you uploaded a newer `eval_random_mask_control_colab.py` to `/content`, that copy is used.

In [ ]:
%pip install -q kagglehub geopy

import shutil, subprocess
from pathlib import Path

GEOCLIP_COMMIT = "976a3d494f43c5f43572348571ba38c28c7877b6"
GEOPROBE_REF = "main"
CONTENT = Path("/content")

def sh(*cmd):
    subprocess.run(cmd, check=True)

if not (CONTENT / "geoclip_source" / "geoclip" / "__init__.py").is_file():
    sh("git", "clone", "https://github.com/Le-Anh-Duy/geo-clip", str(CONTENT / "geoclip_source"))
    sh("git", "-C", str(CONTENT / "geoclip_source"), "checkout", GEOCLIP_COMMIT)

repo = CONTENT / "geoprobe"
if not repo.is_dir():
    sh("git", "clone", "--depth", "1", "--branch", GEOPROBE_REF,
       "https://github.com/Le-Anh-Duy/geoprobe", str(repo))
shutil.copy(repo / "backend" / "app" / "intervention.py", CONTENT)
shutil.copy(repo / "results" / "layer_search" / "proposal_cache" / "proposals.jsonl", CONTENT)

SCRIPT = CONTENT / "eval_random_mask_control_colab.py"
if not SCRIPT.is_file():
    shutil.copy(repo / "experiments" / SCRIPT.name, SCRIPT)
print("script:", SCRIPT)

## 2. Configuration

Run a **smoke test first** (`MAX_IMAGES = "5"`), then set it to `"0"` for the full
2,997 images. Results are checkpointed per image, so re-running the next cell after a
disconnect resumes where it stopped. Change `OUTPUT_NAME` whenever you change the
configuration, otherwise old rows are reused.

In [ ]:
import os

os.environ.update({
    "MAX_IMAGES": "5",                  # "0" = all 2,997 images
    "OUTPUT_NAME": "img2gps3k_random_mask_control_smoke",
    "INTERVENTION_LAYERS": "2,7,12",    # Table 1: layers {2,7,12} under 2/k
    "INTERVENTION_A": str(2 / 3),
    "INTERVENTION_B": str(-2 / 3),
    "RANDOM_REPEATS": "5",
    "SEED": "42",
})

## 3. Run

In [ ]:
import runpy
_ = runpy.run_path(str(SCRIPT), run_name="__main__")

## 4. Read the result

* **`displacement` contrasts** — the xAI question: does intervening on the detected
  regions move the prediction *further* than on an arbitrary region of the same shape?
  `union_vs_shift > 0` with a CI above zero means the regions' content matters.
* **`delta` contrasts** — the accuracy question: positive means `union` reduces error
  more than the control.
* A CI that contains zero for `union_vs_shift` means the displacement in the paper is a
  property of perturbing attention over that much area, not of what the detector found.

In [ ]:
import json
import pandas as pd

summary = json.loads((CONTENT / os.environ["OUTPUT_NAME"] / "summary.json").read_text())
print(f"evaluated: {summary['evaluated_images']}  active: {summary['active_mask_images']}"
      f"  failed: {summary['failed_images']}")

for subset_name, subset in summary["subsets"].items():
    print(); print(f"== {subset_name} ({subset['images']} images)")
    display(pd.DataFrame({
        arm: {
            "mean displacement km": s["mean_displacement_km"],
            "mean clipped Δ km": s["mean_clipped_improvement_km"],
            "mean error km": s["mean_distance_km"],
            "acc@25km": s["accuracy"]["acc_25_km"],
            "acc@200km": s["accuracy"]["acc_200_km"],
        }
        for arm, s in subset["arms"].items()
    }).T.round(3))
    display(pd.DataFrame([
        {"metric": metric, "contrast": pair, "mean diff km": c["mean_difference_km"],
         "ci95 low": c["ci95"][0], "ci95 high": c["ci95"][1], "significant": c["significant"]}
        for metric, pairs in subset["contrasts"].items() for pair, c in pairs.items()
    ]).round(2))

## 5. Save the output

Copy the run directory to Drive (or download the zip) and commit it under `results/` in `geoprobe`.

In [ ]:
out = CONTENT / os.environ["OUTPUT_NAME"]
archive = shutil.make_archive(str(out), "zip", out)
print(archive)
# from google.colab import drive; drive.mount("/content/drive")
# shutil.copy(archive, "/content/drive/MyDrive/")